# Building the applications

One application = one job posting + the pool of CVs that applied to it (~75 CVs). About half of a pool are suitable CVs (same category as the job), the rest are distractors drawn at random from the other categories.

`RHO` (how many times each CV is reused) sets how many applications are built; the other constants only shape a single pool.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../..")
import pandas as pd
import numpy as np
from pathlib import Path

from pipeline.utils.result_utils import render

DATA_DIR = Path("../../datasets/findhr")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
jobs = pd.read_json(DATA_DIR / "source/filtered_jobs_dt.json")
cvs = pd.read_json(DATA_DIR / "source/filtered_cvs_dt.json")

len(jobs), len(cvs)

CVs (`id_c`) and jobs (`id_j`) share the category column `job_category_agg`: a CV is suitable for a job of its own category.

Applications per category:

$$Q = \min\left(\text{round}\left(\frac{RHO \cdot V}{r}\right),\; J\right)$$

`V` = CVs in the category, `J` = its jobs, `r` = expected suitable CVs per pool. A job is never reused, hence the `min` with `J`.

In [ ]:
P_MU      = 0.5      # suitable CVs are ~50% of a pool, on average
P_JITTER  = 0.15     # ...but each pool draws its own share, uniform in 35%..65%
POOL_MU   = 75       # a pool holds ~75 CVs, on average
POOL_SD   = 10       # ...drawn from a normal with this sigma
POOL_CLIP = 30       # ...never shorter than 45 nor longer than 105
RHO       = 12       # target reuse of each CV; sets the number of applications
SEED      = 23
R_MAX     = int(np.floor(POOL_MU * P_MU))  # 37 = expected suitable CVs per pool; only used to compute Q

## Step 1: how many applications per category

`r_exp = min(V, R_MAX)`: a category with fewer than `R_MAX = 37` CVs cannot fill more suitable slots than it has. `Q` depends on the configuration only, not on the random draws.

In [ ]:
cats = sorted(cvs["job_category_agg"].unique())
cv_by_cat  = {c: cvs.loc[cvs["job_category_agg"] == c, "id_c"].tolist() for c in cats}
job_by_cat = {c: jobs.loc[jobs["job_category_agg"] == c, "id_j"].tolist() for c in cats}

In [ ]:
plan = []
for c in cats:
    V = len(cv_by_cat[c])
    J = len(job_by_cat[c])
    r_exp = min(V, R_MAX)  # R_MAX = 37
    Q = min(round(RHO * V / r_exp), J)
    plan.append(dict(category=c, V=V, J=J, r_exp=r_exp, Q=Q))
plan_df = pd.DataFrame(plan).sort_values("V", ascending=False)
plan_df

## Step 2: balanced suitable CVs

Each pool takes the suitable CVs used least so far, ties broken at random (`argsort(used + rng.random(n))`, the jitter is < 1): no CV twice in a pool, and within a category the usage counts differ by at most one.

In [ ]:
def balanced_distinct_pools(items, sizes, rng):
    """One pool per entry of `sizes`, items distinct inside a pool: always take the
    least-used items, ties broken at random. Usage across pools stays balanced to +-1."""
    items = list(items)
    used, pools = np.zeros(len(items)), []
    for r in sizes:
        idx = np.argsort(used + rng.random(len(items)))[:r]  # jitter < 1 -> strict least-used order
        used[idx] += 1
        pools.append([items[i] for i in idx])
    return pools

In [ ]:
render(r"""
\textbf{Step 3 --- building the applications}\par\smallskip\hrule\smallskip
\begin{algorithmic}[1]
\Require CVs $V$ and jobs $J$, each with a category
\Require $\rho{=}12$;\ pool length $\mathcal N(\mu_L{=}75,\ \sigma_L{=}10)$ clipped to $\mu_L \pm \delta_L$, $\delta_L{=}30$;\ suitable share $\mathcal U(\mu_P \pm \epsilon_P)$, $\mu_P{=}0.5$, $\epsilon_P{=}0.15$
\For{each category $c$}
  \State $V_c, J_c \gets$ CVs and jobs of $c$
  \State $r \gets \min(|V_c|, \lfloor \mu_L\mu_P \rfloor)$;\quad $Q \gets \min(\mathrm{round}(\rho|V_c|/r), |J_c|)$
  \State $j_1, \dots, j_Q \gets$ $Q$ distinct jobs sampled from $J_c$;\quad $u(v) \gets 0\ \ \forall v \in V_c$
  \For{$i = 1, \dots, Q$}
    \State $n_i \gets \mathrm{round}(\mathrm{clip}(\mathcal N(\mu_L,\sigma_L),\ \mu_L \pm \delta_L))$;\quad $p_i \sim \mathcal U(\mu_P \pm \epsilon_P)$
    \State $r_i \gets \min(\mathrm{round}(n_i p_i), |V_c|)$;\quad $d_i \gets n_i - r_i$
    \State $R_i \gets$ the $r_i$ CVs of $V_c$ with the lowest $u$, ties broken at random;\quad $u(v) \gets u(v) + 1\ \ \forall v \in R_i$
    \State $D_i \gets$ $d_i$ CVs sampled without replacement from $V \setminus V_c$
    \State \textbf{emit} application $\bigl(j_i,\ \mathrm{shuffle}(R_i \cup D_i),\ R_i\bigr)$
  \EndFor
\EndFor
\end{algorithmic}
\hrule
""")

In [ ]:
rng = np.random.default_rng(SEED)
Q_by_cat = dict(zip(plan_df.category, plan_df.Q))

apps = []
for c in cats:
    V, Q = len(cv_by_cat[c]), Q_by_cat[c]

    sel_jobs = list(rng.choice(job_by_cat[c], size=Q, replace=False))

    # Per-application noise: pool size ~ truncated normal, prevalence ~ uniform.
    pool_sizes = rng.normal(POOL_MU, POOL_SD, Q).clip(POOL_MU - POOL_CLIP, POOL_MU + POOL_CLIP)
    pool_sizes = np.round(pool_sizes).astype(int)
    P_draw = rng.uniform(P_MU - P_JITTER, P_MU + P_JITTER, Q)

    r_sizes = np.minimum(np.round(pool_sizes * P_draw).astype(int), V)  # can't ask for more than V
    d_sizes = pool_sizes - r_sizes

    rel_pools = balanced_distinct_pools(cv_by_cat[c], r_sizes, rng)
    off_pool = cvs.loc[cvs["job_category_agg"] != c, "id_c"].tolist()  # distractors: uniform

    for job, rel, d in zip(sel_jobs, rel_pools, d_sizes):
        dis = list(rng.choice(off_pool, size=int(d), replace=False))
        pool = rel + dis
        rng.shuffle(pool)
        apps.append(dict(
            app_id=len(apps), category=c, id_j=job,
            pool=pool, relevant_cvs=rel,
            n_pool=len(pool), n_relevant=len(rel), n_distractor=len(dis),
        ))

applications = pd.DataFrame(apps)
applications["prevalence"] = applications.n_relevant / applications.n_pool
# the realized reuse is per category, so it goes in the plan
plan_df["reuse"] = (plan_df.category.map(applications.groupby("category").n_relevant.sum())
                    / plan_df.V).round(3)

print("Total applications:", len(applications), "| total pairs:", applications.n_pool.sum())
applications.groupby("category").agg(
    n_apps=("app_id", "size"),
    pool_min=("n_pool", "min"), pool_mean=("n_pool", "mean"), pool_max=("n_pool", "max"),
    prev_min=("prevalence", "min"), prev_mean=("prevalence", "mean"), prev_max=("prevalence", "max"),
).round(3).join(plan_df.set_index("category").reuse)

## Step 4: output

In `datasets/findhr/applications/`: `applications.json` (one application per row, `pool` and `relevant_cvs` as lists), `plan.csv` (one row per category) and `applications_short.csv` (the pools flattened to `(id_j, id_c)` pairs).

In [ ]:
out_dir = DATA_DIR / "applications"
out_dir.mkdir(parents=True, exist_ok=True)

applications.to_json(out_dir / "applications.json", orient="records", indent=2)
plan_df.to_csv(out_dir / "plan.csv", index=False)
applications.head(10)

In [ ]:
applications_short = applications[["id_j", "pool"]].explode("pool").rename(columns={"pool": "id_c"})
applications_short.to_csv(out_dir / "applications_short.csv", index=False)
applications_short.head(10)